In [3]:
# ============================================================
# TRANSFORM CONSTRUCTORS — LOCAL SILVER LAYER
# ============================================================

import os
from pyspark.sql import functions as F

In [4]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:08:26 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:08:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:08:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:08:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:08:27 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:08:27 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:08:27 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [5]:
# -----------------------------------------
# 2. Define Bronze + Silver paths
# -----------------------------------------
bronze_file = f"{BRONZE_ROOT}/constructors/constructors.parquet"
silver_folder = f"{SILVER_ROOT}/constructors"
silver_file = f"{silver_folder}/constructors_silver.parquet"

os.makedirs(silver_folder, exist_ok=True)

print("Bronze:", bronze_file)
print("Silver:", silver_file)

Bronze: /Users/manoharazalki/F1-ANALYTICS/data/bronze/constructors/constructors.parquet
Silver: /Users/manoharazalki/F1-ANALYTICS/data/silver/constructors/constructors_silver.parquet


In [6]:
# -----------------------------------------
# 3. Read Bronze Constructors
# -----------------------------------------
constructors_df = spark.read.parquet(bronze_file)
print("Bronze constructors loaded")
constructors_df.show(5)

Bronze constructors loaded
+-------------+----------+-----------+--------------------+--------------------+--------------------+
|constructorId|      name|nationality|                 url|    ingest_timestamp|         source_file|
+-------------+----------+-----------+--------------------+--------------------+--------------------+
|        adams|     Adams|   american|http://en.wikiped...|2026-06-08 23:07:...|/Users/manoharaza...|
|          afm|       AFM|     german|http://en.wikiped...|2026-06-08 23:07:...|/Users/manoharaza...|
|          ags|       AGS|     french|http://en.wikiped...|2026-06-08 23:07:...|/Users/manoharaza...|
|         alfa|Alfa Romeo|      swiss|http://en.wikiped...|2026-06-08 23:07:...|/Users/manoharaza...|
|   alphatauri|AlphaTauri|    italian|http://en.wikiped...|2026-06-08 23:07:...|/Users/manoharaza...|
+-------------+----------+-----------+--------------------+--------------------+--------------------+
only showing top 5 rows


In [7]:
# -----------------------------------------
# 4. Drop URL column
# -----------------------------------------
constructors_dropped_df = constructors_df.drop("url")

In [8]:
# -----------------------------------------
# 5. Standardize + rename columns
# -----------------------------------------
constructors_renamed_df = (
    constructors_dropped_df
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("name", "constructor_name")
)

print("✔ Constructors renamed preview")
constructors_renamed_df.show(5)

✔ Constructors renamed preview
+--------------+----------------+-----------+--------------------+--------------------+
|constructor_id|constructor_name|nationality|    ingest_timestamp|         source_file|
+--------------+----------------+-----------+--------------------+--------------------+
|         adams|           Adams|   american|2026-06-08 23:07:...|/Users/manoharaza...|
|           afm|             AFM|     german|2026-06-08 23:07:...|/Users/manoharaza...|
|           ags|             AGS|     french|2026-06-08 23:07:...|/Users/manoharaza...|
|          alfa|      Alfa Romeo|      swiss|2026-06-08 23:07:...|/Users/manoharaza...|
|    alphatauri|      AlphaTauri|    italian|2026-06-08 23:07:...|/Users/manoharaza...|
+--------------+----------------+-----------+--------------------+--------------------+
only showing top 5 rows


In [9]:
# -----------------------------------------
# 6. Remove duplicates
# -----------------------------------------
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [10]:
# -----------------------------------------
# 7. Title Case nationality
# -----------------------------------------
constructors_final_df = (
    constructors_distinct_df
        .withColumn("nationality", F.initcap(F.col("nationality")))
)

print("✔ Constructors Silver preview")
constructors_final_df.show(5)

✔ Constructors Silver preview
+--------------+----------------+-----------+--------------------+--------------------+
|constructor_id|constructor_name|nationality|    ingest_timestamp|         source_file|
+--------------+----------------+-----------+--------------------+--------------------+
|         adams|           Adams|   American|2026-06-08 23:07:...|/Users/manoharaza...|
|           afm|             AFM|     German|2026-06-08 23:07:...|/Users/manoharaza...|
|           ags|             AGS|     French|2026-06-08 23:07:...|/Users/manoharaza...|
|          alfa|      Alfa Romeo|      Swiss|2026-06-08 23:07:...|/Users/manoharaza...|
|    alphatauri|      AlphaTauri|    Italian|2026-06-08 23:07:...|/Users/manoharaza...|
+--------------+----------------+-----------+--------------------+--------------------+
only showing top 5 rows


In [11]:
# -----------------------------------------
# 8. Write Silver output (Option C)
# -----------------------------------------
(
    constructors_final_df
        .write
        .mode("overwrite")
        .parquet(silver_file)
)

print(f"✔ Constructors Silver written to: {silver_file}")

✔ Constructors Silver written to: /Users/manoharazalki/F1-ANALYTICS/data/silver/constructors/constructors_silver.parquet


In [12]:
# -----------------------------------------
# 9. Read back for validation
# -----------------------------------------
spark.read.parquet(silver_file).show(5)

+--------------+----------------+-----------+--------------------+--------------------+
|constructor_id|constructor_name|nationality|    ingest_timestamp|         source_file|
+--------------+----------------+-----------+--------------------+--------------------+
|         adams|           Adams|   American|2026-06-08 23:07:...|/Users/manoharaza...|
|           afm|             AFM|     German|2026-06-08 23:07:...|/Users/manoharaza...|
|           ags|             AGS|     French|2026-06-08 23:07:...|/Users/manoharaza...|
|          alfa|      Alfa Romeo|      Swiss|2026-06-08 23:07:...|/Users/manoharaza...|
|    alphatauri|      AlphaTauri|    Italian|2026-06-08 23:07:...|/Users/manoharaza...|
+--------------+----------------+-----------+--------------------+--------------------+
only showing top 5 rows
